# J3 PM | Traitement de données visuelles et textuelles

**Objectifs d'apprentissage :**
- OCR : Extraire le texte d'un document scanné complexe et le corriger via le Prompt Engineering.
- Extraire des images depuis un réseau social (Bluesky).
- Utiliser un modèle de reconnaissance visuelle (Vision API) pour décrire et identifier les éléments d'une image.


In [ ]:
!apt-get update -qq && apt-get install -y tesseract-ocr tesseract-ocr-fra > /dev/null
!pip install pytesseract 


In [ ]:
import requests
import pytesseract
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
from openai import OpenAI

# 1. Coller votre clé API d'OPEN AI
api_key = "sk-votre-cle-secrete-ici"

try:
    client = OpenAI(api_key=api_key)
    print("Client OpenAI initialisé avec succès !")
except Exception as e:
    print("Erreur d'initialisation :", e)


## 1. OCR et correction par LLM : Lire les archives

Souvent en recherche (histoire politique, sociologie), on fait face à des scans de journaux anciens. Tesseract est un outil classique d'OCR, mais il fait des erreurs de mise en page.
Nous allons télécharger la célèbre une de "J'accuse" et utiliser un LLM (Prompt System) pour corriger et formater l'OCR brut.


In [ ]:
# Configuration des en-têtes 
headers = {'User-Agent': 'LilleLMS_SummerSchool/1.0'}

# Téléchargement de la une de L'Aurore (J'accuse)
jaccuse_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/0/04/J%27accuse_-_Gallica_-_page_1.jpg/1280px-J%27accuse_-_Gallica_-_page_1.jpg"
response = requests.get(jaccuse_url, headers=headers)
response.raise_for_status() # Lève une erreur si le téléchargement échoue (ex: erreur 403)
jaccuse_img = Image.open(BytesIO(response.content))
print("Image téléchargée avec succès !")

In [ ]:
# OCR brut : extraction du texte depuis l'image avec Tesseract
raw_text = pytesseract.image_to_string(jaccuse_img, lang='fra')

print("--- Extrait de l'OCR brut ---")
print(raw_text[:400])



Utilisons notre expertise en **Prompt Engineering** pour nettoyer ce texte.


In [ ]:
# Le system prompt définit le rôle de l'assistant
system_prompt = """
Tu es la référence en traitement d'archives. 
Ton rôle est de corriger les coquilles d'un OCR brut et de structurer le texte lisiblement."
"""

# Le user prompt contient la consigne spécifique et les données
user_prompt = f"""
Voici le résultat brut de l'OCR de la une d'un journal.
1. Corrige les erreurs de l'OCR.

Texte OCR brut :
{raw_text[:1500]}
"""


# Appel à l'API OpenAI pour générer la correction
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.1,
    max_tokens=250,
)
print("--- Texte corrigé par le LLM ---")
print(response.choices[0].message.content)



## 2. Analyse systématique d'images (Bluesky)

Nous allons récupérer les dernières images d'un compte politique sur Bluesky pour analyser la scénographie visuelle (présence de symboles).


In [ ]:
# Fonction pour interroger l'API publique de Bluesky et extraire les images d'un compte
def get_bluesky_images(handle, max_images=5):
    # Endpoint de l'API Bluesky pour récupérer le flux d'un auteur
    url = f"https://public.api.bsky.app/xrpc/app.bsky.feed.getAuthorFeed?actor={handle}"
    response = requests.get(url)
    images = []
    
    if response.status_code == 200:
        data = response.json()
        # Parcours des publications (posts) du compte
        for item in data.get('feed', []):
            embed = item.get('post', {}).get('embed', {})
            
            img_list = []
            # Vérifie si le post contient une image directe
            if embed.get('$type') == 'app.bsky.embed.images#view':
                img_list = embed.get('images', [])
            elif embed.get('$type') == 'app.bsky.embed.recordWithMedia#view':
                media = embed.get('media', {})
                if media.get('$type') == 'app.bsky.embed.images#view':
                    img_list = media.get('images', [])
                    
            # Ajout des URLs d'images trouvées à notre liste
            for img in img_list:
                images.append({
                    'author': handle,
                    'image_url': img.get('thumb')
                })
                if len(images) >= max_images:
                    return images
    return images


In [ ]:
# Récupérons les images du Parti Socialiste et d'un compte test riche en images
actor_1 = "partisocialiste.bsky.social"
actor_2 = "jlmelenchon.bsky.social"

# Création d'un DataFrame Pandas avec les résultats combinés
image_df = pd.DataFrame(get_bluesky_images(actor_1, 5) + get_bluesky_images(actor_2, 5))


In [ ]:
# Visualisation rapide
if not image_df.empty:
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    for i, row in image_df.iterrows():
        ax = axes[i // 5, i % 5]
        try:
            ax.imshow(Image.open(BytesIO(requests.get(row['image_url']).content)))
        except:
            pass
        ax.axis('off')
        ax.set_title(row['author'].split('.')[0])
    plt.tight_layout()
    plt.show()


## 3. Description d'images avec un Modèle Multimodal (LLM)

Nous allons utiliser un LLM avec des capacités de vision pour générer une courte description textuelle de chaque image. Nous allons lui demander de limiter sa réponse à 250 caractères maximum pour faciliter l'analyse ultérieure.

In [ ]:
# Fonction pour générer une courte description d'une image via un LLM multimodal
def describe_image(image_url):
    prompt = '''
    Génère une description objective et précise de l'image.
    Décris ce qui est visible uniquement : 
        sujet principal, éléments secondaires, arrière-plan, environnement, 
        positions relatives, couleurs et détails importants.
    Évite toute supposition ou analyse.
    Réponse courte : maximum 250 caractères.
    '''
    
    try:
        # On envoie l'image URL et le texte explicatif au modèle
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "user", "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": image_url}}
                ]}
            ],
            temperature=0.2, # Température basse pour limiter l'hallucination
            max_tokens=150
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Erreur : {str(e)}"


In [ ]:
# Testons sur la première image récupérée
test_url = image_df.iloc[0]['image_url']
print("URL de l'image :", test_url)
print("\n--- Description générée ---")
print(describe_image(test_url))


## 4. Détection d'éléments avec un modèle spécialisé (Zero-Shot Classification)

Demander à un LLM génératif d'inventer des "probabilités" (ex: *"Quelle est la probabilité qu'il y ait un drapeau ?"*) n'a pas de sens méthodologique : le LLM va simplement générer un chiffre plausible (hallucination), et non calculer une vraie probabilité visuelle.

Pour obtenir de **vraies probabilités** (scores de confiance) sur la présence d'objets, nous devons utiliser des modèles de vision spécialisés ou des API comme *Google Cloud Vision*. 

Ici, nous utilisons un modèle de vision ouvert (`CLIP`) via la bibliothèque `transformers` de Hugging Face. Il permet de faire de la *Zero-Shot Classification* : on lui donne une image et des catégories textuelles, et il calcule la probabilité que l'image corresponde à chaque catégorie.

In [ ]:
from transformers import pipeline

# Initialisation du modèle de classification zero-shot (téléchargement lors de la première exécution)
# Ce modèle associe des images à du texte sans avoir été spécifiquement entraîné pour nos catégories
image_classifier = pipeline("zero-shot-image-classification", model="openai/clip-vit-base-patch32")

def detect_symbols(image_url):
    try:
        # Téléchargement de l'image en mémoire pour la passer au modèle
        response = requests.get(image_url)
        img = Image.open(BytesIO(response.content))
        
        # Liste des concepts à tester (CLIP fonctionne mieux en anglais)
        candidate_labels = ["flag", "crowd", "formal suit"]
        
        # Calcul des scores (vraies probabilités issues du réseau de neurones visuel)
        scores = image_classifier(img, candidate_labels=candidate_labels)
        
        # Formatage des résultats dans un dictionnaire
        result_dict = {item['label']: round(item['score'] * 100, 2) for item in scores}
        return result_dict
    except Exception as e:
        return {"erreur": str(e)}



In [ ]:
# Application sur notre jeu de données (Description + Détection de symboles)
results = []
if not image_df.empty:
    for i, row in image_df.iterrows():
        # 1. Génération de la description avec GPT-4o-mini
        description = describe_image(row['image_url'])
        
        # 2. Extraction des probabilités avec CLIP
        symbols = detect_symbols(row['image_url'])
        
        # Compilation des données pour cette image
        row_data = {
            'author': row['author'],
            'description': description
        }
        row_data.update(symbols) # Ajoute les colonnes de probabilités (flag, crowd, formal suit)
        results.append(row_data)

# Affichage du DataFrame final
results_df = pd.DataFrame(results)
display(results_df.head())


### Hack-Time

1. Modifiez le prompt dans `describe_image` pour lui demander d'analyser l'émotion dominante de l'image.
2. Dans `detect_symbols`, remplacez les étiquettes par de nouveaux concepts (ex: `people`, `police`, `smiling`). Observez comment les probabilités changent.

**Discussion :** Pourquoi l'approche *Zero-Shot Classification* est-elle méthodologiquement plus solide pour quantifier des éléments visuels que de demander à un LLM génératif d'estimer la probabilité de ces éléments

## Conclusion 

Nous avons vu comment automatiser l'analyse d'archives numérisées et d'images issues des réseaux sociaux.

L'extraction de texte (OCR) ou la description par un LLM fait gagner un temps précieux. L'utilisation de modèles de vision spécialisés (comme CLIP) permet ensuite de détecter des concepts visuels et d'obtenir des probabilités, plutôt que de se fier à des statistiques hallucinées.

**Points clés à retenir pour vos recherches :**
1. **Complémentarité des outils** : Les LLMs génératifs sont excellents pour nettoyer du texte ou décrire des images, mais ils "hallucinent" des statistiques. Pour des mesures quantitatives, il faut privilégier des modèles spécialisés.
2. **Contrôle des données** : L'IA facilite le redressement des données brutes (archives, réseaux sociaux). Cependant, déléguer cette tâche exige du chercheur une vérification systématique pour garantir que le sens originel des sources n'est pas altéré.
3. **Validité de la mesure** : L'automatisation permet d'analyser massivement des millions de publications. Le défi pour les sciences sociales n'est plus l'extraction technique, mais la *validité* : s'assurer que l'algorithme capte correctement le concept sociologique étudié.